# 01 - Opening I3 Files And Looking Inside

In this notebook you'll learn to:

1. Point Python at an IceCube `.i3.zst` file.
2. Open that file with `dataio.I3File`.
3. Count the different kinds of frames in the file.
4. Print the keys stored inside a Physics frame.


## The files we'll use

IceCube event data is usually split into (1) an event file plus (2) a matching GCD file.

GCD means Geometry, Calibration, and DetectorStatus.

The event file contains event-by-event information; the GCD file describes the detector for that run or simulation set.


In [12]:
from pathlib import Path
from collections import Counter

from icecube import dataio

# Some example simulation data to get us started:
SIM_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133575/Level2_IC86.2019_data_Run00133575_0101_78_503_GCD.i3.zst')
SIM_FILE = Path('/data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst')

# ...along with some real data:
DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

for label, path in [('simulation GCD', SIM_GCD), ('simulation events', SIM_FILE), ('data GCD', DATA_GCD), ('data events', DATA_FILE)]:
    print(f'{label:18s}: exists={path.exists()}')
    print(f'  {path}')


simulation GCD    : exists=True
  /data/exp/IceCube/2020/filtered/level2/0101/Run00133575/Level2_IC86.2019_data_Run00133575_0101_78_503_GCD.i3.zst
simulation events : exists=True
  /data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst
data GCD          : exists=True
  /data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst
data events       : exists=True
  /data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst


## A tiny frame-stop helper

An I3 file is a stream of frames.

Each frame has a "stop", which tells you what kind of frame it is.

In this IceTray environment, Physics frames show up as `P`, DAQ frames as `Q`, Geometry as `G`, Calibration as `C`, and DetectorStatus as `D`.

The function below converts those short labels to names that are easier to read. This is just for clarity :)


In [13]:
def frame_stop(frame):
    """Return a readable name for the frame stop."""
    stop = str(frame.Stop)
    names = {
        'I': 'TrayInfo',
        'G': 'Geometry',
        'C': 'Calibration',
        'D': 'DetectorStatus',
        'Q': 'DAQ',
        'P': 'Physics',
    }
    return names.get(stop, stop)

print('If IceTray gives us P, we will call it:', frame_stop(type('FakeFrame', (), {'Stop': 'P'})()))


If IceTray gives us P, we will call it: Physics


## Count frame types

This next cell opens the experimental event file and reads the first 100 frames.

It counts how many frames of each type appear. We close the file when we are finished.


In [25]:
counts = Counter()
i3_file = dataio.I3File(str(DATA_FILE))  # <-- Just looking at the experimental data for now. You can swap this with SIM_FILE.

for frame_number in range(100):
    if not i3_file.more():
        break
    frame = i3_file.pop_frame()
    counts[frame_stop(frame)] += 1

i3_file.close()

print('Frame types in the first 100 frames:')
for stop, count in counts.items():
    print(f'  {stop:14s} {count}')


Frame types in the first 100 frames:
  TrayInfo       1
  DAQ            31
  Physics        68


## Inspect one Physics frame

A frame is like a dictionary: it contains named objects called "keys".

The code below finds the first Physics frame and prints the first several keys within it.

You'll use routines like this a lot because different files (and data levels) and frames don't have a universal set of keys; you always need to peek inside.

BONUS: Change 'Physics' to 'DAQ' to see what a Q-frame looks like.

In [26]:
physics_frame = None
i3_file = dataio.I3File(str(DATA_FILE))

while i3_file.more():
    frame = i3_file.pop_frame()
    if frame_stop(frame) == 'Physics':
        physics_frame = frame
        break

i3_file.close()

if physics_frame is None:
    raise RuntimeError('No Physics frame was found. Try checking the file path or reading more frames.')

print('Found a Physics frame.')
print(f'This frame has {len(physics_frame.keys())} keys. Here are the first 40:')
for key in list(physics_frame.keys())[:40]:
    try:
        print(' ', key, '->', type(physics_frame[key]).__name__)
    except Exception as error:
        print(' ', key, '-> could not load:', error)

Found a Physics frame.
This frame has 26 keys. Here are the first 40:
  IceTopPulses -> I3RecoPulseSeriesMapUnion
  ReextractedInIcePulsesTimeRange -> I3TimeWindow
  TankPulseMergerExcludedTanks -> I3VectorTankKey
  QFilterMask -> I3FilterResultMap
  FilterMask -> I3FilterResultMap
  CalibratedWaveformRange -> I3TimeWindow
  ReextractedInIcePulses -> I3RecoPulseSeriesMap
  I3SuperDST -> I3SuperDST
  ClusterCleaningExcludedTanks -> I3VectorTankKey
  NFramesIsDifferent -> I3Bool
  InIceDSTPulses -> I3RecoPulseSeriesMapMask
  OfflineIceTopSLCVEMPulses -> I3RecoPulseSeriesMap
  CleanIceTopRawData -> I3DOMLaunchSeriesMap
  ReextractedIceTopPulses -> I3RecoPulseSeriesMap
  InIcePulses -> I3RecoPulseSeriesMapUnion
  SpecialHits -> I3DOMLaunchSeriesMap
  InIceDSTOnlyPulses -> I3RecoPulseSeriesMapMask
  OfflineIceTopHLCTankPulses -> I3RecoPulseSeriesMap
  ReextractedIceTopPulses_SLC -> I3RecoPulseSeriesMap
  IceTopHLCPulseInfo -> could not load: "Deserialization failed for object at frame key '

## Pull out the event header

Most Physics frames contain `I3EventHeader`.

This object stores bookkeeping information such as run number and event number.

This isn't super important but can be useful if, for example, you've processed data from .i3 files into some other stripped-down format (like a .csv or .hdf5 file), but then later need to grab extra info about your events from the original .i3 file.

In [27]:
if 'I3EventHeader' in physics_frame:
    header = physics_frame['I3EventHeader']
    print('Run ID:        ', header.run_id)
    print('Event ID:      ', header.event_id)
    print('Sub-event ID:  ', header.sub_event_id)
    print('Sub-event type:', header.sub_event_stream)
else:
    print('This frame does not contain I3EventHeader.')


Run ID:         133576
Event ID:       2061
Sub-event ID:   0
Sub-event type: NullSplit


---
The "Sub-event type" is for event splitting. There are multiple reasons/cases for this to happen.

Remember: a ***Q-frame*** is the parent detector readout window. It means: “the detector triggered; here is everything recorded around that trigger.”

This can contain one clean physical event, or *several* overlapping/coincident things.

A ***P-frame*** is an analysis-level child frame made from a Q-frame. A splitter takes 1 Q-frame and creates one or more P-frames.

- `event_id` labels the parent (Q) event.
- `sub_event_stream` tells you which splitter/view produced the P-frame, such as `NullSplit` or `InIceSplit`.
- `sub_event_id` distinguishes multiple subevents within the same stream.

In our data, the same event_id=1 will appear as a Q-frame, a `NullSplit` P-frame, and an `InIceSplit` P-frame.

- `NullSplit` means: "do nothing" or rather, “make one P-frame containing the whole DAQ event.”
- `InIceSplit` means: “make P-frame(s) corresponding to in-ice activity" (like multiple muon bundles appearing in far-separated locations of the detector, in which case you'll see 3 P-frames for the same Q-frame: 1 with `NullSplit`, and 2 with `InIceSplit`.

---

## Look for useful keys

This next cell searches the key names for words that will matter later: pulses, filters, reconstructions, and simulation truth.


In [28]:
interesting_words = ['pulse', 'filter', 'spline', 'mpe', 'linefit', 'energy', 'mc', 'primary']

print('Interesting-looking keys:')
for key in physics_frame.keys():
    if any(word in key.lower() for word in interesting_words):
        print(' ', key)


Interesting-looking keys:
  IceTopPulses
  ReextractedInIcePulsesTimeRange
  TankPulseMergerExcludedTanks
  QFilterMask
  FilterMask
  ReextractedInIcePulses
  InIceDSTPulses
  OfflineIceTopSLCVEMPulses
  ReextractedIceTopPulses
  InIcePulses
  InIceDSTOnlyPulses
  OfflineIceTopHLCTankPulses
  ReextractedIceTopPulses_SLC
  IceTopHLCPulseInfo
  IceTopDSTOnlyPulses
  OfflineIceTopHLCVEMPulses
  IceTopDSTPulses


## Practice

1. Change `DATA_FILE` to `SIM_FILE` and rerun the frame-counting cell.
2. Open the GCD file instead of the event file.
3. Use `dataio-shovel` in a terminal and poke around at the contents of these files...

To do this, open a fresh terminal and type:
```
ssh cobalt
<enter your password>
eval $(/cvmfs/icecube.opensciencegrid.org/py3-v4.3.0/setup.sh)
eval /home/ireistr/i3/icetray/build/env-shell.sh  # <-- This is my IceTray build. You may eventually build your own.
dataio-shovel /data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst
```
or switch the /data/... file with any .i3 file. You can even look at both a data or sim file and their respective GCD file at once. Something like:
```
dataio-shovel /data/exp/IceCube/2020/filtered/level2/0101/Run00133575/Level2_IC86.2019_data_Run00133575_0101_78_503_GCD.i3.zst /data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst
```

